# Domain A: Transcriptomic Identity and Neural Coding

This notebook addresses three questions:
- **A1:** Do transcriptomically defined cell types have distinct tuning properties?
- **A2:** Does gene expression predict functional response properties at the single-cell level?
- **A3:** How does transcriptomic identity shape population coding geometry?

**Data:** Zarr multimodal stores with ΔF/F calcium traces (X matrix, cells × trials), stimulus metadata (var), cell-type labels & gene expression (obs).

In [13]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.optimize import curve_fit
from scipy.stats import kruskal, mannwhitneyu, spearmanr, pearsonr
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.decomposition import PCA, NMF
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.manifold import TSNE
from pathlib import Path    
from matplotlib import cm, colors
import warnings
import anndata as ad
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr, zarr_to_df
from functions.visualization import polar_bar_plot

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
MULTIMODAL = Path('../scratch/multimodal_data')
MOUSE_IDS = [f.stem.split('_')[0] for f in MULTIMODAL.glob('*_multimodal_data.zarr')]
SESSIONS = ['session_1', 'session_2', 'session_3']
session_type = ['drifting_gratings']
MOUSE_IDS

['782149h',
 '786297x',
 '790322h',
 '767018h',
 '788406h',
 '767022h',
 '778174x',
 '797371x',
 '755252h']

In [5]:
obs = [adata.obs for adata in adata_list]
obs = pd.concat(obs, ignore_index=True)
# ── Identify gene expression columns ──
META_COLS = {'unique_id', 'mouse_id', 'class_name', 'class_label',
             'class_bootstrapping_probability', 'subclass_name', 'subclass_label',
             'subclass_bootstrapping_probability', 'supertype_name', 'supertype_label',
             'supertype_bootstrapping_probability', 'cluster_name', 'cluster_label',
             'cluster_alias', 'cluster_bootstrapping_probability',
             'x_loc', 'y_loc', 'z_loc'}
GENE_COLS = [c for c in obs.columns if c not in META_COLS]
print(f"Gene expression columns: {len(GENE_COLS)}")

# ── Color palette for subclasses ──
SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])
contrasts = np.array([0.05, 0.1, 0.2, 0.4, 0.8])
tfs = np.array([1, 2, 4, 8, 15])


Gene expression columns: 307


## A3: Population Coding Geometry

How does transcriptomic identity shape population-level stimulus representations? We examine:
- **A3.1** PCA / UMAP of population activity, colored by stimulus & cell type
- **A3.2** Representational Similarity Analysis (RSA) per subclass, with CKA comparison
- **A3.3** Stimulus decoding per subclass and mixed-population decoding

In [60]:
# ══════════════════════════════════════════════════════════════════════
# A3.1  PCA / UMAP of population activity colored by stimulus & cell type
# ══════════════════════════════════════════════════════════════════════
from sklearn.manifold import TSNE
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("⚠️ umap-learn not installed; falling back to t-SNE")

# ── Build trial-averaged stimulus response matrix ──
# Each row = one cell, each column = one stimulus condition (orientation × contrast in blocks 0,2)
# We use the contrast-context blocks for orientation × contrast conditions
var_cb = adata.var.copy()
block_mask = var_cb['stim_block'].isin([0.0, 2.0])

conditions = []
cond_labels = []
for ori in orientations:
    for c in contrasts:
        mask = block_mask & (var_cb['orientation'] == ori) & (var_cb['contrast'] == c)
        tidx = np.where(mask.values)[0]
        if len(tidx) > 0:
            conditions.append(np.nanmean(adata.X[:, tidx], axis=1))
            cond_labels.append(f"ori={int(ori)}°, c={c}")

stim_mat = np.column_stack(conditions)  # cells × conditions
print(f"Stimulus response matrix: {stim_mat.shape} (cells × conditions)")

# Z-score per cell for embedding
stim_z = (stim_mat - stim_mat.mean(axis=1, keepdims=True)) / (stim_mat.std(axis=1, keepdims=True) + 1e-8)
valid_cells_a3 = ~np.isnan(stim_z).any(axis=1)
stim_z_clean = stim_z[valid_cells_a3]
labels_a3 = obs['subclass_name'].values[valid_cells_a3]

# ── PCA ──
pca = PCA(n_components=10)
pcs = pca.fit_transform(stim_z_clean)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PC1 vs PC2, colored by cell type
ax = axes[0]
for sc in present_subclasses:
    m = labels_a3 == sc
    ax.scatter(pcs[m, 0], pcs[m, 1], alpha=0.4, s=12,
               color=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc])
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA — by Cell Type', fontweight='bold')
ax.legend(fontsize=7, markerscale=2)

# Scree plot
ax = axes[1]
ax.bar(range(1, 11), pca.explained_variance_ratio_ * 100, color='steelblue')
ax.plot(range(1, 11), np.cumsum(pca.explained_variance_ratio_) * 100, 'ro-', markersize=5)
ax.set_xlabel('PC')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('PCA Scree Plot', fontweight='bold')

# PC1 vs PC2, colored by preferred orientation
pref_ori_a3 = tuning_df['pref_ori'].values[valid_cells_a3]
ax = axes[2]
sc_ori = ax.scatter(pcs[:, 0], pcs[:, 1], c=pref_ori_a3, cmap='hsv',
                    alpha=0.4, s=12, vmin=0, vmax=180)
plt.colorbar(sc_ori, ax=ax, label='Pref. Orientation (°)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA — by Preferred Orientation', fontweight='bold')

plt.suptitle('A3.1: Population Activity Embeddings', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── UMAP or t-SNE ──
if HAS_UMAP:
    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.3, random_state=42)
    embed_2d = reducer.fit_transform(stim_z_clean)
    method_name = 'UMAP'
else:
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    embed_2d = tsne.fit_transform(stim_z_clean)
    method_name = 't-SNE'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for sc in present_subclasses:
    m = labels_a3 == sc
    ax.scatter(embed_2d[m, 0], embed_2d[m, 1], alpha=0.5, s=15,
               color=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc])
ax.set_title(f'{method_name} — by Cell Type', fontweight='bold')
ax.legend(fontsize=7, markerscale=2)
ax.set_xticks([]); ax.set_yticks([])

ax = axes[1]
sc_ori2 = ax.scatter(embed_2d[:, 0], embed_2d[:, 1], c=pref_ori_a3, cmap='hsv',
                     alpha=0.5, s=15, vmin=0, vmax=180)
plt.colorbar(sc_ori2, ax=ax, label='Pref. Orientation (°)')
ax.set_title(f'{method_name} — by Preferred Orientation', fontweight='bold')
ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f'A3.1: {method_name} Embedding of Population Responses', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Stimulus response matrix: (1037, 40) (cells × conditions)


IndexError: Boolean index has wrong length: 1037 instead of 2824

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# A3.2  Representational Similarity Analysis (RSA) per subclass + CKA
# ══════════════════════════════════════════════════════════════════════
from scipy.spatial.distance import squareform, pdist, correlation

def centered_kernel_alignment(X, Y):
    """CKA with linear kernel between two representations (samples × features)."""
    K = X @ X.T
    L = Y @ Y.T
    # Center kernels
    n = K.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    Kc = H @ K @ H
    Lc = H @ L @ H
    hsic = np.sum(Kc * Lc)
    norm = np.sqrt(np.sum(Kc * Kc) * np.sum(Lc * Lc))
    return hsic / norm if norm > 0 else 0.0

# ── Build trial-averaged response per stimulus condition per subclass ──
# Stimulus conditions: each unique (orientation, contrast) pair in contrast-context blocks
n_conditions = len(cond_labels)

# For RSA we need the same conditions compared across subclasses
# RDM per subclass: pairwise distances between condition-averaged responses
rdms = {}
subclass_stim_mats = {}
for sc in present_subclasses:
    sc_mask = (obs['subclass_name'].values == sc) & valid_cells_a3
    if sc_mask.sum() < 5:
        continue
    # Mean response per condition across cells of this subclass
    sc_stim_vec = stim_mat[sc_mask]  # cells × conditions
    # RDM = pairwise correlation distance between conditions (using cell responses as features)
    rdm = squareform(pdist(sc_stim_vec.T, metric='correlation'))
    rdms[sc] = rdm
    subclass_stim_mats[sc] = sc_stim_vec

print(f"RDMs computed for {len(rdms)} subclasses, {n_conditions} conditions each")

# ── Visualize RDMs ──
n_rdm = len(rdms)
fig, axes = plt.subplots(1, n_rdm, figsize=(4.5 * n_rdm, 4))
if n_rdm == 1:
    axes = [axes]
for ax, sc in zip(axes, rdms):
    im = ax.imshow(rdms[sc], cmap='viridis', vmin=0, vmax=2)
    ax.set_title(SUBCLASS_SHORT[sc], fontweight='bold', color=SUBCLASS_COLORS[sc])
    ax.set_xlabel('Condition')
    ax.set_ylabel('Condition')
plt.suptitle('A3.2: Representational Dissimilarity Matrices (1 - corr)', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=axes, shrink=0.7, label='Correlation Distance')
plt.tight_layout()
plt.show()

# ── CKA comparison between all pairs of subclasses ──
sc_list = list(subclass_stim_mats.keys())
n_sc = len(sc_list)

# We need matched conditions → use the condition-averaged response matrix
# Cell-averaged representation per subclass: conditions × 1 (mean across cells)
# Better: use the full cell × condition matrix, subsample to same size for CKA
min_cells = min(subclass_stim_mats[s].shape[0] for s in sc_list)
rng = np.random.default_rng(42)

cka_matrix = np.zeros((n_sc, n_sc))
n_repeats = 10
for i in range(n_sc):
    for j in range(n_sc):
        cka_vals = []
        for _ in range(n_repeats):
            Xi = subclass_stim_mats[sc_list[i]]
            Xj = subclass_stim_mats[sc_list[j]]
            # Subsample to min_cells for fair comparison
            idx_i = rng.choice(Xi.shape[0], size=min(min_cells, Xi.shape[0]), replace=False)
            idx_j = rng.choice(Xj.shape[0], size=min(min_cells, Xj.shape[0]), replace=False)
            # conditions × subsampled_cells
            cka_vals.append(centered_kernel_alignment(Xi[idx_i].T, Xj[idx_j].T))
        cka_matrix[i, j] = np.mean(cka_vals)

fig, ax = plt.subplots(figsize=(8, 6))
short_labels_cka = [SUBCLASS_SHORT[s] for s in sc_list]
sns.heatmap(cka_matrix, xticklabels=short_labels_cka, yticklabels=short_labels_cka,
            annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1, ax=ax)
ax.set_title('A3.2: CKA Similarity of Stimulus Representations Across Subclasses',
             fontweight='bold')
plt.tight_layout()
plt.show()

# ── RDM correlation matrix (second-order RSA) ──
rdm_vectors = {sc: squareform(rdms[sc]) for sc in sc_list}
rdm_corr = np.zeros((n_sc, n_sc))
for i in range(n_sc):
    for j in range(n_sc):
        rho, _ = spearmanr(rdm_vectors[sc_list[i]], rdm_vectors[sc_list[j]])
        rdm_corr[i, j] = rho

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(rdm_corr, xticklabels=short_labels_cka, yticklabels=short_labels_cka,
            annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax)
ax.set_title('A3.2: Second-Order RSA (Spearman ρ between RDMs)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# A3.3  Stimulus decoding per subclass and mixed-population decoding
# ══════════════════════════════════════════════════════════════════════
from sklearn.model_selection import StratifiedKFold

# ── Prepare single-trial data for decoding ──
# Target: orientation (8 classes), using contrast-context blocks
var_dec = adata.var.copy()
block_dec_mask = var_dec['stim_block'].isin([0.0, 2.0]) & (~var_dec['gray_screen'].astype(bool))
trial_idx_dec = np.where(block_dec_mask.values)[0]

X_all_trials = adata.X[:, trial_idx_dec].T  # trials × cells
y_ori = var_dec.iloc[trial_idx_dec]['orientation'].values.astype(int)

# Remove trials with NaN
valid_trials = ~np.isnan(X_all_trials).any(axis=1)
X_all_trials = X_all_trials[valid_trials]
y_ori = y_ori[valid_trials]
print(f"Decoding data: {X_all_trials.shape[0]} trials × {X_all_trials.shape[1]} cells")
print(f"Orientation classes: {np.unique(y_ori)}")

sc_labels_all = obs['subclass_name'].values
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Decode from each subclass separately ──
decode_results = {}
for sc in present_subclasses:
    sc_cell_idx = np.where(sc_labels_all == sc)[0]
    if len(sc_cell_idx) < 5:
        continue
    X_sc = X_all_trials[:, sc_cell_idx]

    accs = []
    for train_idx, test_idx in skf.split(X_sc, y_ori):
        scaler_d = StandardScaler()
        X_tr = scaler_d.fit_transform(X_sc[train_idx])
        X_te = scaler_d.transform(X_sc[test_idx])
        # LDA
        lda = LinearDiscriminantAnalysis()
        lda.fit(X_tr, y_ori[train_idx])
        accs.append(accuracy_score(y_ori[test_idx], lda.predict(X_te)))
    decode_results[sc] = {'lda_acc': np.mean(accs), 'lda_se': np.std(accs) / np.sqrt(len(accs)),
                          'n_cells': len(sc_cell_idx)}

# ── Decode from ALL cells combined ──
accs_all = []
for train_idx, test_idx in skf.split(X_all_trials, y_ori):
    scaler_d = StandardScaler()
    X_tr = scaler_d.fit_transform(X_all_trials[train_idx])
    X_te = scaler_d.transform(X_all_trials[test_idx])
    lda = LinearDiscriminantAnalysis()
    lda.fit(X_tr, y_ori[train_idx])
    accs_all.append(accuracy_score(y_ori[test_idx], lda.predict(X_te)))
decode_results['All'] = {'lda_acc': np.mean(accs_all), 'lda_se': np.std(accs_all) / np.sqrt(len(accs_all)),
                         'n_cells': X_all_trials.shape[1]}

# ── Decode from excitatory only, then add each inhibitory subclass ──
exc_subclasses = [s for s in present_subclasses if 'Glut' in s]
inh_subclasses = [s for s in present_subclasses if 'Gaba' in s]
exc_cell_idx = np.where(np.isin(sc_labels_all, exc_subclasses))[0]

additive_results = {'Exc only': {'idx': exc_cell_idx}}
for inh_sc in inh_subclasses:
    inh_idx = np.where(sc_labels_all == inh_sc)[0]
    if len(inh_idx) < 3:
        continue
    combined_idx = np.concatenate([exc_cell_idx, inh_idx])
    additive_results[f'Exc + {SUBCLASS_SHORT[inh_sc]}'] = {'idx': combined_idx}

for label, info in additive_results.items():
    X_sub = X_all_trials[:, info['idx']]
    accs_sub = []
    for train_idx, test_idx in skf.split(X_sub, y_ori):
        scaler_d = StandardScaler()
        X_tr = scaler_d.fit_transform(X_sub[train_idx])
        X_te = scaler_d.transform(X_sub[test_idx])
        lda = LinearDiscriminantAnalysis()
        lda.fit(X_tr, y_ori[train_idx])
        accs_sub.append(accuracy_score(y_ori[test_idx], lda.predict(X_te)))
    additive_results[label]['acc'] = np.mean(accs_sub)
    additive_results[label]['se'] = np.std(accs_sub) / np.sqrt(len(accs_sub))

# ══════════════════════════════════════════════════════════════════════
# Visualization
# ══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ── Panel 1: Decoding accuracy by subclass ──
ax = axes[0]
sc_names_dec = [SUBCLASS_SHORT.get(s, s) for s in decode_results]
accs_dec = [decode_results[s]['lda_acc'] for s in decode_results]
ses_dec = [decode_results[s]['lda_se'] for s in decode_results]
colors_dec = [SUBCLASS_COLORS.get(s, '#333333') for s in decode_results]
bars = ax.bar(sc_names_dec, accs_dec, yerr=ses_dec, capsize=4, color=colors_dec, edgecolor='black')
ax.axhline(1/8, ls='--', color='gray', alpha=0.5, label='Chance (1/8)')
ax.set_ylabel('Orientation Decoding Accuracy')
ax.set_title('Decoding per Subclass (LDA)', fontweight='bold')
ax.legend()
ax.tick_params(axis='x', rotation=45)
# Annotate n_cells
for bar, sc in zip(bars, decode_results):
    n = decode_results[sc]['n_cells']
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'n={n}', ha='center', va='bottom', fontsize=8)

# ── Panel 2: Additive inhibitory contribution ──
ax = axes[1]
add_labels = list(additive_results.keys())
add_accs = [additive_results[k]['acc'] for k in add_labels]
add_ses = [additive_results[k]['se'] for k in add_labels]
ax.barh(add_labels, add_accs, xerr=add_ses, capsize=4, color='steelblue', edgecolor='black')
ax.axvline(1/8, ls='--', color='gray', alpha=0.5)
ax.set_xlabel('Decoding Accuracy')
ax.set_title('Excitatory + Inhibitory Subclass', fontweight='bold')

# ── Panel 3: Decoding as function of population size ──
ax = axes[2]
# Subsample cells at different population sizes
pop_sizes = [10, 25, 50, 100, 200, 500, min(1000, X_all_trials.shape[1])]
pop_sizes = [p for p in pop_sizes if p <= X_all_trials.shape[1]]
pop_accs = []
pop_ses_list = []
rng_dec = np.random.default_rng(42)
for ps in pop_sizes:
    rep_accs = []
    for rep in range(5):
        cell_subset = rng_dec.choice(X_all_trials.shape[1], size=ps, replace=False)
        X_sub_ps = X_all_trials[:, cell_subset]
        fold_accs = []
        for train_idx, test_idx in skf.split(X_sub_ps, y_ori):
            scaler_d = StandardScaler()
            X_tr = scaler_d.fit_transform(X_sub_ps[train_idx])
            X_te = scaler_d.transform(X_sub_ps[test_idx])
            lda = LinearDiscriminantAnalysis()
            lda.fit(X_tr, y_ori[train_idx])
            fold_accs.append(accuracy_score(y_ori[test_idx], lda.predict(X_te)))
        rep_accs.append(np.mean(fold_accs))
    pop_accs.append(np.mean(rep_accs))
    pop_ses_list.append(np.std(rep_accs) / np.sqrt(len(rep_accs)))

ax.errorbar(pop_sizes, pop_accs, yerr=pop_ses_list, marker='o', linewidth=2,
            capsize=4, color='darkblue')
ax.axhline(1/8, ls='--', color='gray', alpha=0.5, label='Chance')
ax.set_xlabel('Population Size (# cells)')
ax.set_ylabel('Decoding Accuracy')
ax.set_title('Decoding vs Population Size', fontweight='bold')
ax.set_xscale('log')
ax.legend()

plt.suptitle('A3.3: Stimulus Decoding Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()